# Lição 4 — Chamada de variantes (*Variant Calling*)

Adaptado de [Data Carpentry — Wrangling Genomics, Episódio 4](https://datacarpentry.github.io/wrangling-genomics/04-variant_calling.html).

As células deste notebook rodam em **Bash** (kernel `bash_kernel`), então os comandos são idênticos aos que você digitaria no terminal.

---

## Objetivos

Ao final desta lição você será capaz de:

- **Inspecionar** um arquivo de alinhamento **BAM** (`samtools view`, `samtools flagstat`)
- **Indexar** um BAM (`samtools index`)
- **Chamar variantes** com `bcftools mpileup` e `bcftools call`
- **Filtrar** as variantes chamadas
- **Visualizar** as variantes no alinhamento

## Configuração inicial

Rode a célula abaixo **uma vez** para garantir que estamos na raiz do repositório (onde fica a pasta `data/`). A partir daí, todos os caminhos serão escritos como no terminal do Data Carpentry (`data/...`).

In [ ]:
# Garante que o diretório de trabalho é a raiz do repositório (onde fica a pasta data/).
# Rode esta célula uma vez, no início. É seguro rodá-la mais de uma vez.
[ -d data ] || cd ..
pwd
ls data

## Seção 0 — Preparação (já executada) ⚙️

> **Você não precisa rodar esta seção.** Os comandos abaixo já foram executados para preparar o material, e os resultados já estão no repositório. Estão aqui **apenas para reprodutibilidade** — para mostrar de onde veio o arquivo BAM com que a aula começa.

Esta aula assume que você **já viu o mapeamento de *reads* (*read mapping*)**. Por isso, começamos a partir de um alinhamento pronto. As etapas de preparação foram:

1. **Download** do genoma de referência de *E. coli* e dos *reads* já filtrados (*trimmed*).
2. **Indexação** do genoma de referência para o `bwa` (`bwa index`).
3. **Alinhamento** dos *reads* ao genoma com `bwa mem`, gerando um arquivo **SAM**.
4. **Conversão** SAM → BAM (`samtools view`) e **ordenação** por posição (`samtools sort`).

O resultado é o arquivo `data/bam/SRR2584866.aligned.sorted.bam`, o ponto de partida da nossa aula.

In [ ]:
# ⚙️ PREPARAÇÃO — já executado; NÃO é necessário rodar em aula.
# Mantido comentado apenas para reprodutibilidade (de onde veio o BAM inicial).
#
# --- 1. Download dos dados (feito uma vez) ---
# Genoma de referência de E. coli:
# curl -L -o data/ref_genome/ecoli_rel606.fasta.gz \
#   ftp://ftp.ensemblgenomes.org/pub/bacteria/release-37/fasta/bacteria_0_collection/escherichia_coli_str_k_12_substr_mg1655/dna/Escherichia_coli_str_k_12_substr_mg1655.ASM584v2.dna.chromosome.Chromosome.fa.gz
# gunzip data/ref_genome/ecoli_rel606.fasta.gz
#
# Reads já filtrados (subconjunto do Data Carpentry, via figshare):
#   https://ndownloader.figshare.com/files/14418248  ->  data/trimmed_fastq_small/
#
# --- 2. Indexar a referência para o bwa ---
# bwa index data/ref_genome/ecoli_rel606.fasta
#
# --- 3. Alinhar os reads (bwa mem) -> SAM ---
# bwa mem data/ref_genome/ecoli_rel606.fasta \
#   data/trimmed_fastq_small/SRR2584866_1.trim.sub.fastq \
#   data/trimmed_fastq_small/SRR2584866_2.trim.sub.fastq \
#   > SRR2584866.aligned.sam
#
# --- 4. SAM -> BAM (samtools view) e ordenar por posição (samtools sort) ---
# samtools view -S -b SRR2584866.aligned.sam \
#   | samtools sort -o data/bam/SRR2584866.aligned.sorted.bam -

---

## Seção 1 — Inspecionar o alinhamento

Nossa aula começa aqui. Temos um alinhamento pronto e **ordenado por posição**:
`data/bam/SRR2584866.aligned.sorted.bam`.

O formato **BAM** é a versão **binária e comprimida** do formato **SAM** (texto). Por ser binário, não conseguimos lê-lo diretamente com `head` ou `cat` — usamos `samtools view` para traduzi-lo de volta para texto SAM legível.

Vamos espiar as primeiras linhas de alinhamento:

In [ ]:
# As primeiras linhas de alinhamento (samtools traduz o BAM binário de volta para texto SAM)
samtools view data/bam/SRR2584866.aligned.sorted.bam | head

### Estatísticas gerais do alinhamento

O comando `samtools flagstat` dá um resumo rápido do alinhamento: total de *reads*, quantos foram mapeados, quantos formam pares corretamente alinhados, etc. É uma boa checagem de sanidade **antes** de chamar variantes.

In [ ]:
# Resumo do alinhamento: total de reads, mapeados, pares corretos, etc.
samtools flagstat data/bam/SRR2584866.aligned.sorted.bam